In [1]:
## importing required libraries
import os
import shutil
import random
from tqdm.notebook import tqdm

In [2]:
train_path_img = "./yolo_data/images/train/"
train_path_label = "./yolo_data/labels/train/"
val_path_img = "./yolo_data/images/val/"
val_path_label = "./yolo_data/labels/val/"
test_path = "./yolo_data/test"

In [ ]:

'''
Split the dataset into train and test and creates the train.txt and test.tx with
the respective path of the images in each folder
'''

def train_test_split(path,neg_path=None, split = 0.2):
    print("------ PROCESS STARTED -------")


    files = list(set([name[:-4] for name in os.listdir(path)])) ## removing duplicate names i.e. counting only number of images


    print (f"--- This folder has a total number of {len(files)} images---")
    random.seed(42)
    random.shuffle(files)

    test_size = int(len(files) * split)
    train_size = len(files) - test_size

    ## creating required directories

    os.makedirs(train_path_img, exist_ok = True)
    os.makedirs(train_path_label, exist_ok = True)
    os.makedirs(val_path_img, exist_ok = True)
    os.makedirs(val_path_label, exist_ok = True)


    ### ----------- copying images to train folder
    for filex in tqdm(files[:train_size]):
      if filex == 'classes':
          continue  
      file_tif = path + filex + '.tif'
      file_txt = path + filex + '.txt'
        
      if not os.path.exists(file_tif):
          print(f"Error: File {file_tif} not found.")
      if not os.path.exists(file_txt):
          print(f"Error: File {file_txt} not found.")
          
      shutil.copy2(path + filex + '.tif',f"{train_path_img}/" + filex + '.tif' )
      shutil.copy2(path + filex + '.txt', f"{train_path_label}/" + filex + '.txt')



    print(f"------ Training data created with 80% split {len(files[:train_size])} images -------")

    if neg_path:
        neg_images = list(set([name[:-4] for name in os.listdir(neg_path)])) ## removing duplicate names i.e. counting only number of images
        for filex in tqdm(neg_images):
            shutil.copy2(neg_path+filex+ ".tif", f"{train_path_img}/" + filex + '.tif')

        print(f"------ Total  {len(neg_images)} negative images added to the training data -------")

        print(f"------ TOTAL Training data created with {len(files[:train_size]) + len(neg_images)} images -------")



    ### copytin images to validation folder
    for filex in tqdm(files[train_size:]):
      if filex == 'classes':
          continue
      # print("running")
      shutil.copy2(path + filex + '.tif', f"{val_path_img}/" + filex + '.tif' )
      shutil.copy2(path + filex + '.txt', f"{val_path_label}/" + filex + '.txt')

    print(f"------ Testing data created with a total of {len(files[train_size:])} images ----------")

    print("------ TASK COMPLETED -------")

## spliting the data into train-test and creating train.txt and test.txt files
# train_test_split('/content/drive/MyDrive/custom_notebooks/yolo_data/')

#### The path to the folder having the custom data is set here ####
### for label_tag
train_test_split('./Data/') ### without negative images # IMAGE files with annotations IMP
# train_test_split('./data/','./negative_images/') ### if you want to feed negative images

------ PROCESS STARTED -------
--- This folder has a total number of 522 images---


  0%|          | 0/418 [00:00<?, ?it/s]

------ Training data created with 80% split 418 images -------


  0%|          | 0/104 [00:00<?, ?it/s]

------ Testing data created with a total of 104 images ----------
------ TASK COMPLETED -------


In [9]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.163  Python-3.10.0 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
Setup complete  (24 CPUs, 127.1 GB RAM, 378.3/931.5 GB disk)


In [10]:
import torch
torch.cuda.empty_cache()

In [6]:
import torch
print(torch.cuda.is_available())

True


In [7]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo12m.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(task="detect" ,data="./dataset_custom.yaml", epochs=500, imgsz=1024, project="./Training_Results", name="YoloV12M_500_1024", batch=8, device=0)

Ultralytics 8.3.163  Python-3.10.0 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset_custom.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=YoloV12M_500_102424, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pos

train: Scanning D:\Model_Training\yolo_data\labels\train... 503 images, 0 backgrounds, 0 corrupt: 100%|██████████| 503/503 [00:00<00:00, 1249.75it/s]

train: New cache created: D:\Model_Training\yolo_data\labels\train.cache


val: Fast image access  (ping: 0.00.0 ms, read: 2934.5342.9 MB/s, size: 8976.1 KB)


val: Scanning D:\Model_Training\yolo_data\labels\val... 189 images, 0 backgrounds, 0 corrupt: 100%|██████████| 189/189 [00:00<00:00, 1743.93it/s]

...

    118/500      20.8G      3.959      5.294      1.907         26       1024: 100%|██████████| 63/63 [00:14<00:00,  4.29it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:01<00:00,  8.73it/s]

                   all        189        445          0          0          0          0



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    119/500      20.8G      4.001      5.254      1.902         31       1024:   6%|▋         | 4/63 [00:01<00:16,  3.55it/s]


KeyboardInterrupt: 

In [8]:
model = YOLO("D:/Model_Training/Training_Results/YoloV12M_500_102424/weights/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95(B)
metrics.box.map50  # map50(B)

Ultralytics 8.3.163  Python-3.10.0 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
YOLOv12m summary (fused): 169 layers, 20,105,302 parameters, 0 gradients, 66.9 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 3134.1135.4 MB/s, size: 8791.9 KB)


val: Scanning D:\Model_Training\yolo_data\labels\val.cache... 189 images, 0 backgrounds, 0 corrupt: 100%|██████████| 189/189 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:02<00:00,  5.28it/s]


                   all        189        445          0          0          0          0
                 water         92        107          0          0          0          0
                   oil        152        338          0          0          0          0
Speed: 0.2ms preprocess, 7.5ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs\detect\val3


np.float64(0.0)

In [ ]:
import torch
print(torch.__version__)

In [ ]:
!yolo task=detect mode=predict model=./Training_Results/Yolov11M_1000_1024/weights/best.pt conf=0.5 source=./Test_Images

Ultralytics 8.3.163  Python-3.10.0 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
YOLO11m summary (fused): 125 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs



Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python310\Scripts\yolo.exe\__main__.py", line 7, in <module>
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\ultralytics\cfg\__init__.py", line 983, in entrypoint
    getattr(model, mode)(**overrides)  # default args from model
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\ultralytics\engine\model.py", line 555, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\ultralytics\engine\predictor.py", line 

In [ ]:
from ultralytics import YOLO

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available. Using GPU.")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

model = YOLO("./Training_Results/Yolov11M_1000_1024/weights/best.pt")

# Export the model to TensorRT format
model.export(format="engine")  # creates 'yolo11n.engine'

In [ ]:
# Load the exported TensorRT model
tensorrt_model = YOLO("./Training_Results/Yolov11M_1000_1024/weights/best.engine")

# Run inference
tensorrt_model.predict(
    source='./Test_Images',
    conf=0.5,
    imgsz=1024,  # ✅ match training/input size
    save=True
)

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading Training_Results\Yolov11M_1000_1024\weights\best.engine for TensorRT inference...


ModuleNotFoundError: No module named 'tensorrt'

In [ ]:
# Load the exported TensorRT model
model = YOLO("./Training_Results/Yolov11M_1000_1024/weights/best.pt")

# Run inference
model.predict(
    source='./Test_Images',
    conf=0.5,
    imgsz=1024,  # ✅ match training/input size
    save=True
)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
torch.cuda.is_available()

In [ ]:
print(torch._dynamo.list_backends())